# LeetCode #156 — Binary Tree Upside Down: lật ngược cây theo nhánh trái

**Mức độ:** Medium · **Chủ đề:** Tree, Depth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def upsideDownBinaryTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
```

Mục tiêu của notebook:

1. Hiểu dạng cây đặc biệt của đề: một **xương sống trái**, mỗi nút phải là **lá** có anh em trái.
2. Quy tắc lật cho **một** nút cha và hai con, rồi áp dụng dọc xương sống.
3. Hai cách: **đệ quy** (lật từ dưới lên) và **lặp** (giống đảo danh sách liên kết).
4. Lỗi kinh điển: quên cắt con trỏ cũ → tạo **chu trình**.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân. Lật ngược cây và trả về gốc mới. Với mỗi nút cha có con trái `L` và con phải `R`:

1. Con trái `L` trở thành **cha** mới.
2. Nút cha cũ trở thành **con phải** của `L`.
3. Con phải cũ `R` trở thành **con trái** của `L`.

Áp dụng từng tầng một.

Đề **đảm bảo**: mọi nút phải đều có một anh em (nút trái cùng cha) và **không có con**.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 10]`.
- `1 <= Node.val <= 10`
- Mọi nút phải trong cây đều có anh em và không có con.

---

## Ví dụ

```text
Đầu vào [1,2,3,4,5]          Đầu ra [4,5,2,null,null,3,1]

        1                             4
       / \                           / \
      2   3                         5   2
     / \                               / \
    4   5                             3   1
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [6]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [7]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [8]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [9]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [10]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.3 Áp dụng vào bài này

Cây của đề thực chất là một "xương sống" đi theo con trái (như một **danh sách liên kết**), mỗi nút có thể kèm một lá phải. Lật cây = đổi quan hệ cha–con dọc xương sống, làm giống **đảo ngược danh sách liên kết** với ba con trỏ `prev`, `cur`, `nxt` — nhớ cắt liên kết cũ để không tạo chu trình.

# 3. Nền tảng

In [11]:
from typing import List, Optional
from collections import deque
import random, sys


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ danh sách theo tầng (kiểu LeetCode, None là trống)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def to_level_list(root):
    """Cây → danh sách theo tầng (bỏ None thừa ở cuối). Báo lỗi nếu không còn là cây."""
    out, q, seen = [], deque([root]), set()
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        if id(node) in seen:
            raise ValueError("gặp lại một nút — có chu trình hoặc nút có hai cha!")
        seen.add(id(node))
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

print(to_level_list(build_tree([1, 2, 3, 4, 5])))

[1, 2, 3, 4, 5]


## Hình dạng cây hợp lệ

Vì nút phải luôn là lá, chỉ nút **trái** mới có con → cây là một **xương sống trái** `s0 = root, s1, s2, …, sk`, mỗi `s_i` (trừ `sk`) có thể có thêm một lá phải `r_i`:

```text
s0
| \
s1  r0
| \
s2  r1
|
…
sk
```

Sau khi lật: gốc mới là `sk`; với mỗi `i`, `s_{i+1}.left = r_i`, `s_{i+1}.right = s_i`; `s0` thành lá.

# 4. Baseline: làm đúng theo mô tả — thu xương sống rồi nối lại

Rõ ràng, dễ kiểm chứng; dùng `O(h)` bộ nhớ cho danh sách. Ta dùng làm đáp án đối chiếu.

In [12]:
def upside_down_by_spine(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if not root:
        return None
    spine, rights = [], []
    node = root
    while node:
        spine.append(node)
        rights.append(node.right)
        node = node.left
    for node in spine:
        node.left = node.right = None
    for i in range(len(spine) - 1):
        spine[i + 1].left = rights[i]
        spine[i + 1].right = spine[i]
    return spine[-1]

print(to_level_list(upside_down_by_spine(build_tree([1, 2, 3, 4, 5]))))

[4, 5, 2, None, None, 3, 1]


# 5. Ý tưởng chính 1: đệ quy

Lật **cây con trái** trước (gốc mới của cả cây chính là gốc mới của cây con trái — nút trái nhất). Sau đó `root.left` (giờ là **lá** dưới cùng bên phải của cây đã lật) nhận:

```text
root.left.left  = root.right
root.left.right = root
root.left = root.right = None      # cắt liên kết cũ!
```

In [13]:
def upside_down_recursive(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if not root or not root.left:
        return root
    new_root = upside_down_recursive(root.left)
    root.left.left = root.right
    root.left.right = root
    root.left = root.right = None
    return new_root

print(to_level_list(upside_down_recursive(build_tree([1, 2, 3, 4, 5]))))

[4, 5, 2, None, None, 3, 1]


# 6. Ý tưởng chính 2: lặp — giống đảo danh sách liên kết

Đi xuống xương sống trái. Mỗi nút `cur` nhận: con trái = **con phải của nút cha** (lưu trong `prev_right`), con phải = **nút cha** (`prev`).

```text
prev = prev_right = None
cur = root
while cur:
    nxt = cur.left            # lưu trước khi ghi đè
    cur.left = prev_right
    prev_right = cur.right    # lưu trước khi ghi đè
    cur.right = prev
    prev, cur = cur, nxt
return prev
```

Giống `#206 Reverse Linked List` với `next` = `left`, chỉ thêm việc "chuyển" con phải xuống một tầng.

In [14]:
def upside_down_iterative(root: Optional[TreeNode]) -> Optional[TreeNode]:
    prev = prev_right = None
    cur = root
    while cur:
        nxt = cur.left
        cur.left = prev_right
        prev_right = cur.right
        cur.right = prev
        prev, cur = cur, nxt
    return prev

print(to_level_list(upside_down_iterative(build_tree([1, 2, 3, 4, 5]))))

[4, 5, 2, None, None, 3, 1]


# 7. Mô phỏng (bản lặp)

In [15]:
def trace_iterative(values):
    root = build_tree(values)
    prev = prev_right = None
    cur = root
    v = lambda n: None if n is None else n.val
    while cur:
        nxt = cur.left
        cur.left = prev_right
        prev_right = cur.right
        cur.right = prev
        print(f"cur = {cur.val}: left ← {v(cur.left)}, right ← {v(cur.right)} | prev_right mới = {v(prev_right)}")
        prev, cur = cur, nxt
    print("Kết quả:", to_level_list(prev))
    return prev

_ = trace_iterative([1, 2, 3, 4, 5])

cur = 1: left ← None, right ← None | prev_right mới = 3
cur = 2: left ← 3, right ← 1 | prev_right mới = 5
cur = 4: left ← 5, right ← 2 | prev_right mới = None
Kết quả: [4, 5, 2, None, None, 3, 1]


# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def upsideDownBinaryTree(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        prev = prev_right = None
        cur = root
        while cur:
            nxt = cur.left
            cur.left = prev_right
            prev_right = cur.right
            cur.right = prev
            prev, cur = cur, nxt
        return prev

Bộ sinh cây hợp lệ ngẫu nhiên: xương sống trái độ dài `1..k`, mỗi nút có con trái thì có thể có thêm lá phải.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def random_valid_tree(max_spine=6):
    k = random.randint(1, max_spine)
    spine = [TreeNode(random.randint(1, 10)) for _ in range(k)]
    for i in range(k - 1):
        spine[i].left = spine[i + 1]
        if random.random() < 0.6:
            spine[i].right = TreeNode(random.randint(1, 10))
    return spine[0]


def clone(node):
    return None if node is None else TreeNode(node.val, clone(node.left), clone(node.right))


sol = Solution()
check("Ví dụ 1", to_level_list(sol.upsideDownBinaryTree(build_tree([1, 2, 3, 4, 5]))), [4, 5, 2, None, None, 3, 1])
check("Rỗng", to_level_list(sol.upsideDownBinaryTree(None)), [])
check("Một nút", to_level_list(sol.upsideDownBinaryTree(build_tree([1]))), [1])
check("Không có phải", to_level_list(sol.upsideDownBinaryTree(build_tree([1, 2, None, 3]))), [3, None, 2, None, 1])

random.seed(156)
for _ in range(3000):
    t = random_valid_tree()
    exp = to_level_list(upside_down_by_spine(clone(t)))
    got1 = to_level_list(sol.upsideDownBinaryTree(clone(t)))
    got2 = to_level_list(upside_down_recursive(clone(t)))
    assert got1 == exp == got2
print("3000 cây hợp lệ ngẫu nhiên: lặp, đệ quy và bản theo mô tả khớp nhau")

Ví dụ 1          PASS  [4, 5, 2, None, None, 3, 1]
Rỗng             PASS  []
Một nút          PASS  [1]
Không có phải    PASS  [3, None, 2, None, 1]
3000 cây hợp lệ ngẫu nhiên: lặp, đệ quy và bản theo mô tả khớp nhau


# 9. Vì sao đúng? Độ phức tạp?

**Bản lặp — bất biến:** trước khi xử lý `s_i`, `prev = s_{i-1}` đã là gốc của phần đã lật gồm `s_0..s_{i-1}` và các lá `r_0..r_{i-2}`; `prev_right = r_{i-1}` (lá phải của cha `s_{i-1}`, chưa gắn). Bước xử lý gán `s_i.left = r_{i-1}`, `s_i.right = s_{i-1}` — đúng quy tắc lật — và chuyển `r_i` sang `prev_right`. Kết thúc, `prev = s_k` là gốc mới. Riêng `s_0` nhận `left = right = None` (vì `prev`, `prev_right` ban đầu là `None`) → không còn liên kết cũ.

**Bản đệ quy:** quy nạp theo độ dài xương sống; sau khi lật cây con trái, `root.left` (= `s_1`) là lá phải dưới cùng của cây đã lật, ta gắn hai con cho nó và cắt liên kết của `root`.

```text
Time:  O(n)
Space: O(1) bản lặp; O(h) bản đệ quy (h = độ dài xương sống)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Đệ quy nhưng quên cắt `root.left`, `root.right`

In [18]:
def wrong_no_cut(root):
    if not root or not root.left:
        return root
    new_root = wrong_no_cut(root.left)
    root.left.left = root.right
    root.left.right = root                         # root vẫn trỏ xuống root.left → chu trình
    return new_root

try:
    print(to_level_list(wrong_no_cut(build_tree([1, 2, 3, 4, 5]))))
except ValueError as e:
    print("Lỗi:", e)

Lỗi: gặp lại một nút — có chu trình hoặc nút có hai cha!


`2.right = 1` nhưng `1.left` vẫn là `2` → `2 → 1 → 2 → …`.

## Lỗi 2: Trả về `root` thay vì gốc mới

In [19]:
def wrong_return_root(root):
    if not root or not root.left:
        return root
    wrong_return_root(root.left)                   # gốc mới bị bỏ qua
    root.left.left = root.right
    root.left.right = root
    root.left = root.right = None
    return root                                    # SAI: root giờ là lá

print("Sai :", to_level_list(wrong_return_root(build_tree([1, 2, 3, 4, 5]))), "| Đúng:", to_level_list(sol.upsideDownBinaryTree(build_tree([1, 2, 3, 4, 5]))))

Sai : [1] | Đúng: [4, 5, 2, None, None, 3, 1]


## Lỗi 3: Bản lặp ghi đè `cur.right` trước khi lưu

In [20]:
def wrong_iter_order(root):
    prev = prev_right = None
    cur = root
    while cur:
        nxt = cur.left
        cur.left = prev_right
        cur.right = prev                           # SAI: ghi đè trước…
        prev_right = cur.right                     # …rồi mới lưu → mất r_i
        prev, cur = cur, nxt
    return prev

try:
    print(to_level_list(wrong_iter_order(build_tree([1, 2, 3, 4, 5]))))
except ValueError as e:
    print("Lỗi:", e)

Lỗi: gặp lại một nút — có chu trình hoặc nút có hai cha!


`prev_right` nhận nhầm **nút cha** thay vì lá phải → ở tầng dưới, nút cha bị gắn **hai lần** (làm con trái của nút cháu và con phải của nút con) → mất lá `3`, `5`, và nút `1` có **hai cha** (`4.left` và `2.right`) — không còn là cây.

## Lỗi 4: Áp dụng cho cây bất kỳ

Thuật toán chỉ đúng với cây thỏa điều kiện đề (nút phải là lá). Nếu nút phải có con, cấu trúc con của nó vẫn được giữ — nhưng định nghĩa "lật" khi đó không còn rõ ràng. Bài tập 5 kiểm tra điều kiện đầu vào.

# 11. Mẹo nhận diện pattern

- **Cây thực chất là một chuỗi** (xương sống trái) → xử lý như danh sách liên kết: lặp với `prev`, `cur`, `nxt`.
- **Đổi quan hệ cha–con** → nhớ **cắt** liên kết cũ để không tạo chu trình.

Bài liên quan: **#206 Reverse Linked List**, **#226 Invert Binary Tree**, **#114 Flatten Binary Tree to Linked List**, **#1666 Change the Root of a Binary Tree**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lật cây `[1, 2, 3, 4, 5, None, None, 6, 7]` (xương sống `1-2-4-6`, lá phải `3, 5, 7`). Viết kết quả theo tầng.

## Bài 2 — Điền chỗ trống (đệ quy)

In [21]:
def exercise_2_upside_down(root: Optional[TreeNode]) -> Optional[TreeNode]:
    if not root or not root.left:
        return root
    # TODO: lật cây con trái; gắn root.right và root vào root.left; cắt liên kết của root
    pass


# assert to_level_list(exercise_2_upside_down(build_tree([1, 2, 3, 4, 5]))) == [4, 5, 2, None, None, 3, 1]
# assert to_level_list(exercise_2_upside_down(build_tree([1]))) == [1]

## Bài 3 — Biến thể: đếm chiều dài xương sống trái

Trả về số nút trên xương sống trái (từ gốc đi trái tới hết). Đây cũng là **độ cao** của cây sau khi lật tính theo nhánh phải.

In [22]:
def exercise_3_spine_length(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_3_spine_length(build_tree([1, 2, 3, 4, 5])) == 3
# assert exercise_3_spine_length(None) == 0

## Bài 4 — Biến thể: lật ngược trở lại

Cây sau khi lật có **xương sống phải** (`sk → … → s0`), mỗi nút có thể có **lá trái**. Viết hàm đưa về cây ban đầu. (Gợi ý: đối xứng với bản lặp, đổi vai trò `left`/`right`.)

In [23]:
def exercise_4_flip_back(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# assert to_level_list(exercise_4_flip_back(build_tree([4, 5, 2, None, None, 3, 1]))) == [1, 2, 3, 4, 5]
# assert exercise_4_flip_back(None) is None

## Bài 5 — Mini challenge: kiểm tra điều kiện đầu vào

Trả về `True` nếu mọi nút phải trong cây đều **có anh em trái** và **không có con**.

In [24]:
def exercise_5_is_valid_input(root: Optional[TreeNode]) -> bool:
    # TODO
    pass


# assert exercise_5_is_valid_input(build_tree([1, 2, 3, 4, 5])) is True
# assert exercise_5_is_valid_input(build_tree([1, None, 2])) is False
# assert exercise_5_is_valid_input(build_tree([1, 2, 3, None, None, 4])) is False
# assert exercise_5_is_valid_input(None) is True

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_iterative` sau khi tự làm.

**Bài 3**
- Mức 1: `while node: count += 1; node = node.left`.

**Bài 4**
- Mức 1: bản lặp: `nxt = cur.right; cur.right = prev_left; prev_left = cur.left; cur.left = prev; prev, cur = cur, nxt`.
- Mức 2: kiểm tra: `flip_back(upside_down(t))` phải bằng `t`.

**Bài 5**
- Mức 1: duyệt mọi nút; nếu `node.right` tồn tại thì cần `node.left` tồn tại và `node.right` là lá.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
_ = trace_iterative([1, 2, 3, 4, 5, None, None, 6, 7])

cur = 1: left ← None, right ← None | prev_right mới = 3
cur = 2: left ← 3, right ← 1 | prev_right mới = 5
cur = 4: left ← 5, right ← 2 | prev_right mới = 7
cur = 6: left ← 7, right ← 4 | prev_right mới = None
Kết quả: [6, 7, 4, None, None, 5, 2, None, None, 3, 1]


## Đáp án Bài 2

In [26]:
def answer_2_upside_down(root: Optional[TreeNode]) -> Optional[TreeNode]:
    return upside_down_recursive(root)

assert to_level_list(answer_2_upside_down(build_tree([1, 2, 3, 4, 5]))) == [4, 5, 2, None, None, 3, 1]
assert to_level_list(answer_2_upside_down(build_tree([1]))) == [1]
print("Bài 2 PASS (giống upside_down_recursive ở mục 5)")

Bài 2 PASS (giống upside_down_recursive ở mục 5)


## Đáp án Bài 3

In [27]:
def answer_3_spine_length(root: Optional[TreeNode]) -> int:
    count = 0
    while root:
        count += 1
        root = root.left
    return count

assert answer_3_spine_length(build_tree([1, 2, 3, 4, 5])) == 3
assert answer_3_spine_length(None) == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_flip_back(root: Optional[TreeNode]) -> Optional[TreeNode]:
    prev = prev_left = None
    cur = root
    while cur:
        nxt = cur.right
        cur.right = prev_left
        prev_left = cur.left
        cur.left = prev
        prev, cur = cur, nxt
    return prev

assert to_level_list(answer_4_flip_back(build_tree([4, 5, 2, None, None, 3, 1]))) == [1, 2, 3, 4, 5]
assert answer_4_flip_back(None) is None
for _ in range(3000):
    t = random_valid_tree()
    before = to_level_list(t)
    assert to_level_list(answer_4_flip_back(sol.upsideDownBinaryTree(t))) == before
print("Bài 4 PASS (lật rồi lật lại = cây ban đầu)")

Bài 4 PASS (lật rồi lật lại = cây ban đầu)


## Đáp án Bài 5

In [29]:
def answer_5_is_valid_input(root: Optional[TreeNode]) -> bool:
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        if node.right:
            if not node.left or node.right.left or node.right.right:
                return False
        if node.left:
            stack.append(node.left)
    return True

assert answer_5_is_valid_input(build_tree([1, 2, 3, 4, 5])) is True
assert answer_5_is_valid_input(build_tree([1, None, 2])) is False
assert answer_5_is_valid_input(build_tree([1, 2, 3, None, None, 4])) is False
assert answer_5_is_valid_input(None) is True
for _ in range(1000):
    assert answer_5_is_valid_input(random_valid_tree())
print("Bài 5 PASS")

Bài 5 PASS


Chú ý: ta chỉ cần đẩy con **trái** vào ngăn xếp — con phải hợp lệ là lá nên không cần duyệt tiếp; con phải không hợp lệ đã trả `False` ngay.

# 15. Tổng kết

> Cây của đề là một xương sống trái kèm lá phải. Lật = với mỗi nút trên xương sống: `left ← lá phải của cha`, `right ← cha`. Lặp như đảo danh sách liên kết (`prev`, `prev_right`, `cur`, `nxt`), `O(n)` / `O(1)`; hoặc đệ quy và **nhớ cắt** `root.left = root.right = None`.

---

## Checklist trước khi nộp

- [ ] Lưu `cur.left` và `cur.right` trước khi ghi đè?
- [ ] Đệ quy: cắt liên kết cũ của `root`?
- [ ] Trả về **gốc mới** (nút trái nhất)?
- [ ] Cây rỗng / một nút?